# പാഠം 10 - ഉത്പാദനത്തിൽ എഐ ഏജന്റുകൾ

ഈ പാഠത്തിൽ നിങ്ങൾ പഠിക്കും **ഉത്പാദന മാതൃകകൾ** എഐ ഏജന്റുകൾക്കായി Microsoft Agent Framework (Python) ഉപയോഗിച്ച്. നാം ഉൾക്കൊള്ളുന്നവ:

- **നിരീക്ഷണക്ഷമത** — ഏജന്റ് ഇടപെടലുകൾക്ക് സമയം അളക്കലും ലോഗിംഗ് ചേർക്കലും
- **മൂല്യനിർണയം** — ഒരു മൂല്യനിർണയ ഏജന്റ് ഉപയോഗിച്ച് പ്രതികരണത്തിന്റെ ഗുണനിലവാരം സ്കോർ ചെയ്യൽ
- **ചെലവ് നിയന്ത്രണം** — ടോക്കൺ ഒപ്റ്റിമൈസേഷനും മോഡൽ തിരഞ്ഞെടുപ്പും സംബന്ധിച്ച തന്ത്രങ്ങൾ

ഈ സാഹചര്യമാണ് ഒരു **യാത്രാ ഏജന്റ്**, ഉപയോക്താക്കളെ യാത്രകൾ പദ്ധതിയിടുന്നതിൽ സഹായിക്കുന്നതും, അതിന് മേൽ നിരീക്ഷണവും മൂല്യനിർണയവും ഘടിപ്പിച്ചിട്ടുള്ളത്.


## ക്രമീകരണം


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## പ്രൊഡക്ഷൻ പരിഗണനകൾ

പ്രോട്ടോടൈപ്പുകളിൽ നിന്ന് പ്രൊഡക്ഷനിലേക്കായി എഐ ഏജന്റുകൾ മാറ്റുമ്പോൾ മൂന്ന് അടിസ്ഥാന സ്തംഭങ്ങൾക്ക് സൂക്ഷ്മ ശ്രദ്ധ നൽകേണ്ടതുണ്ട്:

1. **നിരീക്ഷണക്ഷമത** — ഏജന്റ് എന്ത് ചെയ്യുന്നു, അത് എത്ര സമയം എടുക്കുന്നു, ഏത് ടൂളുകൾ അത് വിളിക്കുന്നു എന്നിവ കാണാൻ കഴിവ് വേണം. ട്രേസിംഗും ലോഗിംഗും ഇല്ലാതെ പ്രൊഡക്ഷൻ പ്രശ്നങ്ങൾ ഡീബഗ് ചെയ്യുന്നത് പ്രായോഗികമായി അസാധ്യമാണ്.

2. **മൂല്യനിർണ്ണയം** — ഓട്ടോമേറ്റഡ് ഗുണനിലവാര പരിശോധനകൾ ഏജന്റിന്റെ പ്രതികരണങ്ങൾ കാലാകാലങ്ങളിൽ കൃത്യവും പൂര്‍ണവും സഹായകരവുമായവയെന്നതായќе ഉറപ്പുവരുത്തും. ഒരു മൂല്യനിർണ്ണയ ഏജന്റ് നിർവചിച്ച മാനദണ്ഡങ്ങൾ പ്രകാരം പ്രതികരണങ്ങൾക്ക് സ്കോർ നൽകാൻ കഴിയും.

3. **ചെലവ് നിയന്ത്രണം** — ടോക്കൺ ഉപയോഗം ചെലവിനെ നേരിട്ട് ബാധിക്കുന്നു. പ്രോംപ്റ്റ് ഒപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ് എന്നിവ പോലുള്ള തന്ത്രങ്ങൾ ഗുണമേനം നഷ്ടപ്പെടുത്താതെ ചെലവുകൾ നിയന്ത്രണത്തിൽ വെയ്ക്കാൻ സഹായിക്കുന്നു.


## ഒരു നിരീക്ഷിക്കാവുന്ന ഏജന്റ് നിർമ്മാണം

ഞങ്ങൾ യാത്രാ ടൂളുകൾ നിർവചിച്ച് ഏജന്റിന്റെ വിളി ടൈമിങോടെ പൊതിച്ച്, അതിലൂടെ നാം വിലംബം നിരീക്ഷിക്കാനാകും. ഉൽപ്പാദനത്തിൽ നിങ്ങൾ OpenTelemetry അല്ലെങ്കിൽ സമാനമായ ഒരു ട്രേസിങ് ബാക്ക്‌എൻഡുമായി ഇണയിക്കും.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## മൂല്യനിർണയ മാതൃകകൾ

ഉത്പാദനത്തിലുണ്ടാകുന്ന സാധാരണമായ ഒരു മാതൃക രണ്ടാം ഏജന്റിനെ **മൂല്യനിർണയകനായി** ഉപയോഗിക്കുകയാണ്. മൂല്യനിർണയകൻ പ്രാഥമിക ഏജന്റിന്റെ പ്രതികരണം സമഗ്രത, കൃത്യത, സഹായകത എന്നിവ പോലുള്ള മുൻകൂട്ടി നിർവചിച്ച മാനദണ്ഡങ്ങളുടെ അടിസ്ഥാനത്തിൽ വിലയിരുത്തുകയും സ്കോർ ചെയ്യുകയും ചെയ്യുന്നു.

ഇത് സാധ്യമാക്കുന്നു:
- ഉത്തരം ഉപയോക്താക്കൾക്ക് എത്തുന്നതിന് മുമ്പ് ഓട്ടോമേറ്റഡ് ഗുണനിലവാര പരിശോധനകൾ
- പ്രോംപ്റ്റുകളും മോഡലുകളും മാറുമ്പോൾ റിഗ്രഷൻ കണ്ടെത്തൽ
- കാലക്രമേണ ഏജന്റിന്റെ പ്രകടനം നിരന്തരം നിരീക്ഷിക്കൽ


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ചെലവ് നിയന്ത്രണ തന്ത്രങ്ങൾ

ഉത്പാദന AI ഏജന്റുകൾക്ക് ചെലവുകൾ നിയന്ത്രിക്കുക നിർണായകമാണ്. പ്രധാന തന്ത്രങ്ങൾ ഇവയാണ്:

| Strategy | Description |
|---|---|
| **Prompt optimization** | സിസ്റ്റം നിർദ്ദേശങ്ങൾ സംക്ഷിപ്തമായി വയ്ക്കുക. ഇൻപുട്ട് ടോക്കൺകൾ കുറക്കാൻ ആവർത്തനാത്മകമായ കണ്ടെക്സ് നീക്കംചെയ്യുക. |
| **Model selection** | തിരിച്ചറിവ് അല്ലെങ്കിൽ എക്സ്ട്രാക്ഷൻ പോലുള്ള ലളിതപ്പെട്ട ജോലിൾക്കായി ചെറിയ, കുറഞ്ഞ വിലയുള്ള മോഡലുകൾ (ഉദാ. GPT-4o-mini) ഉപയോഗിക്കുക, സങ്കീർണ്ണമായ റീസണിംഗ്‌ക്ക് വലിയ മോഡലുകൾ സംരക്ഷിച്ചു വെക്കുക. |
| **Caching** | ടൂൾ ഫലങ്ങളും പതിവായി വരുന്ന ക്വറിയുകളും കാഷ് ചെയ്യുക, അതിലൂടെ ആവർത്തനമായ API കോൾകൾ ഒഴിവാക്കുക. |
| **Token budgets** | അനാകാംഷയായി ദൈർഘ്യമേറിയ പ്രതികരണങ്ങൾ തടയാൻ `max_tokens` പരിധികൾ നിശ്ചയിക്കുക. |
| **Batching** | സാധ്യമായിടത്ത് പല ഉപയോക്തൃ ക്വറി കളെയും ഒരു sèl API കോൾ ആയി ഗ്രൂപ്പ് ചെയ്യുക. |

പ്രായോഗികമായി, ഒരു ഘട്ടാനുസൃത സമീപനം ഫലപ്രദമാണ്: ലളിതമായ അഭ്യർത്ഥനകൾ ദ്രുതവും കുറഞ്ഞ ചെലവുള്ള മോഡലിലേക്ക് നയിച്ച്, സങ്കീർണ്ണമായ ക്വറിയുകൾ മാത്രമാണ് കൂടുതൽ ശേഷിയുള്ള മോഡലിലേക്ക് ഉയർത്തേണ്ടത്.


## സംക്ഷേപം

ഈ പാഠത്തിൽ നിങ്ങൾ എങ്ങനെ ചെയ്യണം എന്ന് പഠിച്ചു:

1. **നിരീക്ഷണശേഷി ചേർക്കുക** ഏജന്റ് ഇടപെടലുകളിൽ സമയമാനം மற்றும் ലോഗിംഗ് ഉപയോഗിച്ച്, ട്രേസിംഗിനും മോണിറ്ററിംഗിനും വേണ്ടി അടിസ്ഥാനമായ ഘടന ഒരുക്കുന്നു.
2. **ഏജന്റ് പ്രതികരണങ്ങൾ അവലോകനം ചെയ്യുക** പൂർണ്ണത, കൃത്യത, സഹായപ്രദത എന്നിവയ്ക്ക് സ്കോർ നൽകുന്ന ഒരു എവാല്യുവേറ്റർ ഏജന്റ് ഉപയോഗിച്ച് സ്വയമേവ.
3. **ചെലവുകൾ നിയന്ത്രിക്കുക** പ്രോംപ്റ്റ് ഒപ്റ്റിമൈസേഷൻ, മോഡൽ തിരഞ്ഞെടുപ്പ്, കാഷിംഗ്, ടോക്കൺ ബഡ്ജറ്റുകൾ എന്നിവ വഴി.

ഈ പ്രൊഡക്ഷൻ മാതൃകകൾ നിങ്ങളുടെ AI ഏജന്റുകൾ സ്കെയിലിൽ വിശ്വസനീയവും അളക്കാവുന്നതുമായതും ചെലവ് കാര്യക്ഷമവുമായതുമാക്കാൻ സഹായിക്കുന്നു.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
അസ്വീകരണ കുറിപ്പ്:
ഈ ഡോക്യുമെന്റ് AI വിവർത്തന സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച്译ചെയ്തതാണ്. നാം കൃത്യതയ്ക്ക് പരിശ്രമിച്ചാലും, യാന്ത്രിക വിവർത്തനങ്ങളിൽ പിശകുകൾ അല്ലെങ്കിൽ അകൃത്യതകൾ ഉണ്ടാകാമെന്ന് ദയവായി ശ്രദ്ധിക്കുക. സ്വദേശീയ ഭാഷയിലെ മൂല ഡോക്യുമെന്റ് പ്രാമാണികമായ സ്രോതസ്സായി പരിഗണിക്കണം. നിർണായക വിവരങ്ങൾക്ക് പ്രൊഫഷണൽ മനുഷ്യ വിവർത്തനം ശുപാർശ ചെയ്യപ്പെടുന്നു. ഈ വിവർത്തനം ഉപയോഗിച്ചതിലൂടെ സംഭവിക്കുന്ന ഏതെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
